# Multitask discrete flow matching: per-token mixture path for inpainting (2D checkerboard)

The discrete counterpart of the multitask Hadamard stochastic interpolant notebook. It starts from the uniform-source discrete flow matching notebook (Shaul et al. setup, mixture path, $\kappa_t=t^2$) and gives every token its own time. One probability denoiser, trained once, then does unconditional generation, inpainting with any observed set, and token-by-token generation.

| | continuous (Hadamard SI) | discrete (this notebook) |
|---|---|---|
| per-coordinate "time" | $\alpha\in[0,1]^d$, $\alpha_i=1$ noise | $t\in[0,1]^D$, $t^i=1$ data; $\kappa^i=\kappa(t^i)=(t^i)^2$ |
| interpolant | $\alpha\odot x_0+(1-\alpha)\odot x_1$ | $X_t^i=x_1^i$ with prob. $\kappa^i$, else $x_0^i$ (uniform) |
| learned | $\eta(\alpha,x)=\mathbb E[x_0-x_1\mid I=x]$ | factorized posterior $p_{1\mid t}^i(\cdot\mid x)$, given the vector $t$ |
| loss | $\lVert\hat\eta-(x_0-x_1)\rVert^2$ | cross-entropy (Shaul eq. 11) or per-token generalized KL (eq. 37) |
| velocity | $\dot\alpha\odot\hat\eta$ | $u^i=\dfrac{\dot\kappa^i}{1-\kappa^i}\big[p^i_{1\mid t}(y\mid x)-\delta_{x^i}(y)\big]$, $\dot\kappa^i=\kappa'(t^i)\,\dot t^i$ |
| known coordinate | $\alpha=0,\ \dot\alpha=0$ | $t^i=1,\ \dot t^i=0$: rate 0, never jumps |
| pending coordinate | $\alpha=1$ | $t^i=0,\ \dot t^i=0$: stays a uniform token |
| SDE term | $\epsilon=\bar\epsilon\,\alpha$ | corrector with $c^i=\bar c\,\kappa^i(1-\kappa^i)$ |

**Why the uniform source.** With a mask source, a token's value already says whether it is clean, so the posterior given the observed tokens does not depend on time, and a scalar-time model inpaints exactly by clamping. With a uniform source, a clamped observed token looks like noise to a scalar-time model: its posterior treats that token as noise with probability $1-\kappa_t$, and only the late steps, where it trusts the token again, correct this. Telling the model each token's time removes the mismatch. That is the discrete version of feeding $\alpha$ to $\hat\eta$.

Library pieces that assume one scalar time per sample (`MixtureDiscreteProbPath.sample`, `MixturePathGeneralizedKL`, `MixtureDiscreteEulerSolver`) are re-implemented in cells below as per-token subclasses. The scheduler, `categorical` and `ModelWrapper` are used as they are.

References: Shaul et al., *Flow Matching with General Discrete Paths: A Kinetic-Optimal Perspective* (arXiv 2412.03487); Negrel et al., *Multitask Learning with Stochastic Interpolants* (arXiv 2508.04605); https://github.com/facebookresearch/flow_matching/blob/main/examples/2d_discrete_flow_matching.ipynb

In [ ]:
!git clone https://github.com/facebookresearch/flow_matching.git

In [ ]:
%cd flow_matching

!pip install -e .

## Imports and init device

In [ ]:
import time
from dataclasses import dataclass
from math import ceil

import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn, Tensor

# flow_matching
from flow_matching.path import MixtureDiscreteProbPath
from flow_matching.path.path_sample import DiscretePathSample
from flow_matching.path.scheduler import PolynomialConvexScheduler
from flow_matching.loss import MixturePathGeneralizedKL
from flow_matching.solver import MixtureDiscreteEulerSolver
from flow_matching.solver.solver import Solver
from flow_matching.utils import ModelWrapper, categorical

In [ ]:
if torch.cuda.is_available():
    device = 'cuda:0'
    print('Using gpu')
else:
    device = 'cpu'
    print('Using cpu.')

torch.manual_seed(42)

## Dataset

In [ ]:
def inf_train_gen(n_grid_points: int = 128, batch_size: int = 200, device: str = "cpu") -> Tensor:
    assert n_grid_points % 4 == 0, "number of grid points has to be divisible by 4"

    n_grid_points = n_grid_points // 4

    x1 = torch.randint(low=0, high=n_grid_points * 4, size=(batch_size,), device=device)
    samples_x2 = torch.randint(low=0, high=n_grid_points, size=(batch_size,), device=device)

    x2 = (
        samples_x2
        + 2 * n_grid_points
        - torch.randint(low=0, high=2, size=(batch_size,), device=device) * 2 * n_grid_points
        + (torch.floor(x1 / n_grid_points) % 2) * n_grid_points
    )

    x_end = 1.0 * torch.cat([x1[:, None], x2[:, None]], dim=1)

    return x_end.long()


def in_support(x: Tensor, n_grid_points: int = 128) -> float:
    '''Fraction of samples on the checkerboard (8 of the 16 blocks). Target = 1, uniform noise ~ 0.5.'''
    b = n_grid_points // 4
    return (((x[:, 0] // b) + (x[:, 1] // b)) % 2 == 0).float().mean().item()


def block_tv(x: Tensor, n_grid_points: int = 128) -> float:
    '''Total variation between generated and true block masses (true: 1/8 on each of the 8 allowed blocks).'''
    b = n_grid_points // 4
    idx = (x[:, 0] // b) * 4 + (x[:, 1] // b)
    m = torch.bincount(idx, minlength=16).float() / len(x)
    i, j = torch.meshgrid(torch.arange(4), torch.arange(4), indexing='ij')
    true = (((i + j) % 2 == 0).float() / 8).flatten().to(x.device)
    return 0.5 * (m - true).abs().sum().item()


print("target on-support fraction (should be 1):", in_support(inf_train_gen(128, 100000)))

## Task distribution $\nu$: one time per token

Your notebook draws one $t\sim U[0,1-\epsilon]$ per sample. Here each sample gets a vector $t\in[0,1]^D$ from a mixture:

* **diagonal** $t\,\mathbf 1$: exactly your current training, i.e. unconditional generation;
* **independent** $t^i\sim U[0,1-\epsilon]$;
* **mask-like** $t^i\in\{1,\ t,\ 0\}$ (known / being generated / not yet generated): inpainting and sequential generation.

Known tokens ($t^i=1$) are clean, so they carry no loss. Every positive mixture has the same minimizer (the true posterior); the mixture only decides how much capacity each task gets. `task_mix = (1.0, 0.0, 0.0)` in the training cell is your original diagonal-only training, for comparison.

In [ ]:
def sample_token_times(bs: int, d: int, epsilon: float = 1e-3, mix=(0.3, 0.2, 0.5), mask_probs=(0.4, 0.4, 0.2)) -> Tensor:
    '''Per-token times t in [0,1]^d: diagonal | independent uniform | mask-like (1 known / t active / 0 pending).'''
    t    = torch.rand(bs, 1, device=device) * (1 - epsilon)
    kind = torch.multinomial(torch.tensor(mix), bs, replacement=True).to(device)[:, None]
    diag = t.expand(bs, d)
    unif = torch.rand(bs, d, device=device) * (1 - epsilon)
    cat  = torch.multinomial(torch.tensor(mask_probs), bs * d, replacement=True).view(bs, d).to(device)
    mask = torch.where(cat == 0, torch.ones_like(unif), torch.where(cat == 1, diag, torch.zeros_like(unif)))
    return torch.where(kind == 0, diag, torch.where(kind == 1, unif, mask))

## Model: your MLP with a per-token time input

The only change is the time embedding. It now acts on each token's own time and is concatenated to that token's embedding, instead of one scalar time per sample.

In [ ]:
# Activation class
class Swish(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x: Tensor) -> Tensor:
        return torch.sigmoid(x) * x

# Model class
class MLP(nn.Module):
    def __init__(
        self, input_dim: int = 128, time_dim: int = 1, hidden_dim=128, length=2):
        super().__init__()
        self.input_dim = input_dim
        self.time_dim = time_dim
        self.hidden_dim = hidden_dim

        self.time_embedding = nn.Linear(1, time_dim)                    # applied to each token's time
        self.token_embedding = torch.nn.Embedding(self.input_dim, hidden_dim)

        self.main = nn.Sequential(
            Swish(),
            nn.Linear((hidden_dim + time_dim) * length, hidden_dim),
            Swish(),
            nn.Linear(hidden_dim, hidden_dim),
            Swish(),
            nn.Linear(hidden_dim, hidden_dim),
            Swish(),
            nn.Linear(hidden_dim, self.input_dim * length),
        )

    def forward(self, x, t):
        # x: [B, N] tokens, t: [B, N] per-token times
        t = self.time_embedding(t.unsqueeze(-1))                        # [B, N, time_dim]
        x = self.token_embedding(x)                                     # [B, N, hidden_dim]

        B, N, d = x.shape
        h = torch.cat([x, t], dim=-1).reshape(B, -1)
        h = self.main(h)

        h = h.reshape(B, N, self.input_dim)

        return h

## Per-token mixture path, losses and solver

**Path.** $P(X_t^i=x_1^i)=\kappa(t^i)$, otherwise $X_t^i=x_0^i$, independently per token. Known tokens ($t^i=1$) are always $x_1^i$; pending tokens ($t^i=0$) are always $x_0^i$.

**Losses.**
* Cross-entropy $-\sum_i \log p^i_{1\mid t}(x_1^i\mid X_t)$ over tokens that are not known (Shaul eq. 11). It is path-independent, like the continuous $\lVert\hat\eta-(x_0-x_1)\rVert^2$.
* Per-token generalized KL (eq. 37) with jump coefficient $\lambda^i=\dot\kappa(t^i)/(1-\kappa(t^i))$, which is infinite on known tokens, so they are excluded.

Both have the same minimizer. The default is the generalized KL, as in your original notebook; switch `loss_type` to `"ce"` for cross-entropy.

In [ ]:
class HadamardMixtureDiscreteProbPath(MixtureDiscreteProbPath):
    '''MixtureDiscreteProbPath with one time per token: t has the shape of x_1 instead of [batch].'''

    def assert_sample_shape(self, x_0: Tensor, x_1: Tensor, t: Tensor):
        assert x_0.shape == x_1.shape, f"x_0 and x_1 must have the same shape. Got {x_0.shape} and {x_1.shape}."
        assert t.shape == x_1.shape, f"Per-token times must have the shape of x_1 {tuple(x_1.shape)}. Got {tuple(t.shape)}."

    def sample(self, x_0: Tensor, x_1: Tensor, t: Tensor) -> DiscretePathSample:
        self.assert_sample_shape(x_0=x_0, x_1=x_1, t=t)

        sigma_t = self.scheduler(t).sigma_t                             # P(X_t^i = x_0^i) = 1 - kappa(t^i), per token

        source_indices = torch.rand(size=x_1.shape, device=x_1.device) < sigma_t
        x_t = torch.where(condition=source_indices, input=x_0, other=x_1)

        return DiscretePathSample(x_t=x_t, x_1=x_1, x_0=x_0, t=t)

In [ ]:
class HadamardMixtureCrossEntropy(nn.Module):
    '''Cross-entropy on the factorized posterior, averaged over tokens that are not known (t < 1).'''

    def forward(self, logits: Tensor, x_1: Tensor, x_t: Tensor, t: Tensor) -> Tensor:
        ce = F.cross_entropy(logits.transpose(1, 2), x_1, reduction='none')    # [B, N]
        weight = (t < 1).to(ce.dtype)
        return (ce * weight).sum() / weight.sum().clamp_min(1.0)


class HadamardMixtureGeneralizedKL(MixturePathGeneralizedKL):
    '''MixturePathGeneralizedKL with a per-token jump coefficient d kappa(t^i) / (1 - kappa(t^i)); known tokens excluded.'''

    def forward(self, logits: Tensor, x_1: Tensor, x_t: Tensor, t: Tensor) -> Tensor:
        log_p_1t = torch.log_softmax(logits, dim=-1)
        log_p_1t_x1 = torch.gather(log_p_1t, dim=-1, index=x_1.unsqueeze(-1)).squeeze(-1)
        p_1t_xt = torch.gather(torch.exp(log_p_1t), dim=-1, index=x_t.unsqueeze(-1)).squeeze(-1)

        scheduler_output = self.path.scheduler(t)
        jump_coefficient = torch.where(
            t < 1,
            scheduler_output.d_alpha_t / (1 - scheduler_output.alpha_t).clamp_min(1e-12),
            torch.zeros_like(t),
        )
        delta_x1_xt = (x_t == x_1).to(log_p_1t.dtype)

        loss = -jump_coefficient * (p_1t_xt - delta_x1_xt + (1 - delta_x1_xt) * log_p_1t_x1)

        if self.reduction == "mean":
            return torch.mean(loss)
        elif self.reduction == "sum":
            return torch.sum(loss)
        elif self.reduction == "none":
            return loss
        else:
            raise ValueError(f"{self.reduction} is not a valid value for reduction")

## Train

In [ ]:
source_distribution = "uniform"
loss_type = "gkl"           # "gkl" (per-token generalized KL, as in your notebook) or "ce" (cross-entropy)
task_mix = (0.3, 0.2, 0.5)  # diagonal / independent / mask-like; (1.0, 0.0, 0.0) = your original diagonal-only training
time_dim = 1                # per-token time features; your notebook used 1

# training arguments
lr = 0.001
batch_size = 4096
iterations = 30001
print_every = 3000

vocab_size = 128
hidden_dim = 128
dim = 2

epsilon = 1e-3

# probability denoiser model init
probability_denoiser = MLP(input_dim=vocab_size, time_dim=time_dim, hidden_dim=hidden_dim, length=dim).to(device)

# per-token convex path
scheduler = PolynomialConvexScheduler(n=2.0)
path = HadamardMixtureDiscreteProbPath(scheduler=scheduler)

# init optimizer
optim = torch.optim.Adam(probability_denoiser.parameters(), lr=lr)

loss_fn = HadamardMixtureCrossEntropy() if loss_type == "ce" else HadamardMixtureGeneralizedKL(path=path)

# train
start_time = time.time()

losses = []
for i in range(iterations):
    optim.zero_grad()

    # sample data: (X_0, X_1) ~ uniform x data
    x_1 = inf_train_gen(n_grid_points=vocab_size, batch_size=batch_size, device=device)
    x_0 = torch.randint_like(x_1, high=vocab_size)

    # sample one time per token (was: one time per sample)
    t = sample_token_times(batch_size, dim, epsilon=epsilon, mix=task_mix)

    # sample the per-token probability path
    path_sample = path.sample(t=t, x_0=x_0, x_1=x_1)

    # loss
    logits = probability_denoiser(x=path_sample.x_t, t=path_sample.t)
    loss = loss_fn(logits=logits, x_1=x_1, x_t=path_sample.x_t, t=path_sample.t)

    # optimizer step
    loss.backward()
    optim.step()

    losses.append(loss.item())
    if (i+1) % print_every == 0:
        elapsed = time.time() - start_time
        print('| iter {:6d} | {:5.2f} ms/step | loss {:8.3f} '
              .format(i+1, elapsed*1000/print_every, loss.item()))
        start_time = time.time()

In [ ]:
k = 200
plt.figure(figsize=(8, 4))
plt.plot(losses, alpha=0.3, label='per step')
plt.plot(np.arange(k - 1, len(losses)), np.convolve(losses, np.ones(k) / k, mode='valid'), label=f'moving average ({k})')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title(f'Training loss ({loss_type}); the floor is > 0')
plt.legend()
plt.grid(True)
plt.show()

## Paths through the cube of token times, and the per-token Euler solver

`TokenPath` puts each token in one state:

| state | time $t^i(s)$ | $\dot t^i$ | |
|---|---|---|---|
| known | 1 | 0 | observed or already generated: never jumps |
| active | $s$ | 1 | generated along this path |
| pending | 0 | 0 | stays a uniform token until its own stage |

The solver is Shaul et al.'s Algorithm 1 (the library's `MixtureDiscreteEulerSolver`) with per-token rates. At each step it samples $X_1^i\sim p^i_{1\mid t}$ and jumps to it with probability $1-e^{-h\lambda^i}$, where $\lambda^i=\dot\kappa^i/(1-\kappa^i)$. On the last step, active tokens are set to $X_1^i$; known tokens are never touched.

**Corrector** (optional, the library's `div_free` term). It adds $c\,(\hat u-\check u)$ with the model-free backward velocity $\check u=\frac{\dot\kappa}{\kappa}[\delta_{x^i}-p]$. With $c^i=\bar c\,\kappa^i(1-\kappa^i)$ the extra rates are $\bar c\,\dot\kappa^i\big[\kappa^i\delta_{x_1^i}+(1-\kappa^i)p\big]$: bounded at both ends, and zero on known and pending tokens. This mirrors $\epsilon=\bar\epsilon\,\alpha$ in the continuous SDE.

In [ ]:
@dataclass
class TokenPath:
    known:  Tensor   # [N] or [B, N]; 1 = observed / already generated (time 1)
    active: Tensor   # [N] or [B, N]; 1 = generated along this path (time s); the rest is pending (time 0)

    def time(self, s):     return self.known + self.active * s
    def time_dot(self, s): return self.active


def unconditional_path(d):
    return TokenPath(known=torch.zeros(d, device=device), active=torch.ones(d, device=device))


def inpainting_path(obs):
    '''obs: [N] or [B, N], 1 on observed tokens.'''
    obs = obs.to(device).float()
    return TokenPath(known=obs, active=1 - obs)


def sequential_paths(order, d):
    '''Generate token order[0], then order[1] given order[0], ...'''
    paths = []
    for k, i in enumerate(order):
        known  = torch.zeros(d, device=device)
        active = torch.zeros(d, device=device)
        for j in order[:k]:
            known[j] = 1
        active[i] = 1
        paths.append(TokenPath(known, active))
    return paths

In [ ]:
class HadamardMixtureEulerSolver(Solver):
    '''MixtureDiscreteEulerSolver with one time per token, following a TokenPath.'''

    def __init__(self, model: ModelWrapper, path: MixtureDiscreteProbPath, vocabulary_size: int, source_distribution_p: Tensor = None):
        super().__init__()
        self.model = model
        self.path = path
        self.vocabulary_size = vocabulary_size
        self.source_distribution_p = source_distribution_p

    @torch.no_grad()
    def sample(self, x_init: Tensor, token_path: TokenPath, step_size: float, corrector: float = 0.0,
               time_fn=None, n_snapshots: int = 0, dtype_categorical: torch.dtype = torch.float32):
        '''
        corrector: c_bar >= 0, the corrector strength (c^i = c_bar * kappa^i (1 - kappa^i)).
        time_fn:   optional s -> times fed to the model instead of token_path.time(s) (used for the naive-clamp ablation).
        '''
        if corrector > 0:
            assert self.source_distribution_p is not None, "The corrector needs the source distribution p."

        x_t = x_init.clone()
        n_steps = ceil(1.0 / step_size)
        ts = torch.linspace(0.0, 1.0, n_steps + 1, device=x_t.device)
        active = token_path.active.expand_as(x_t).bool()
        snapshots = [x_t.clone()]

        for i in range(n_steps):
            s, h = ts[i], ts[i + 1] - ts[i]
            t_tok = token_path.time(s).expand_as(x_t)
            t_in = t_tok if time_fn is None else time_fn(s).expand_as(x_t)

            # Sample x_1 ~ p_1|t( . | x_t)
            p_1t = self.model(x=x_t, t=t_in)
            x_1 = categorical(p_1t.to(dtype=dtype_categorical))

            if i == n_steps - 1:
                x_t = torch.where(active, x_1, x_t)                     # final step: only active tokens take x_1
            else:
                scheduler_output = self.path.scheduler(t_tok)
                k_t = scheduler_output.alpha_t
                d_k_t = scheduler_output.d_alpha_t * token_path.time_dot(s).expand_as(x_t)

                rate = torch.where(active, d_k_t / (1 - k_t).clamp_min(1e-12), torch.zeros_like(k_t))
                delta_1 = F.one_hot(x_1, num_classes=self.vocabulary_size).to(k_t.dtype)
                u = rate.unsqueeze(-1) * delta_1

                if corrector > 0:
                    p_0 = self.source_distribution_p[(None,) * x_t.dim()]
                    c = torch.where(active, corrector * d_k_t, torch.zeros_like(k_t)).unsqueeze(-1)
                    u = u + c * ((1 - k_t).unsqueeze(-1) * p_0 + k_t.unsqueeze(-1) * delta_1)

                # Set u_t(x_t | x_t, x_1) = 0
                delta_t = F.one_hot(x_t, num_classes=self.vocabulary_size).bool()
                u = torch.where(delta_t, torch.zeros_like(u), u)

                # Jump with probability 1 - exp(-h * intensity)
                intensity = u.sum(dim=-1)
                mask_jump = torch.rand(size=x_t.shape, device=x_t.device) < 1 - torch.exp(-h * intensity)
                if mask_jump.sum() > 0:
                    x_t[mask_jump] = categorical(u[mask_jump].to(dtype=dtype_categorical))

            if n_snapshots and (i + 1) % max(1, n_steps // n_snapshots) == 0:
                snapshots.append(x_t.clone())

        return (x_t, snapshots) if n_snapshots else x_t


class WrappedModel(ModelWrapper):
    def forward(self, x: torch.Tensor, t: torch.Tensor, **extras):
        return torch.softmax(self.model(x, t, **extras), dim=-1)


wrapped_probability_denoiser = WrappedModel(probability_denoiser)
solver = HadamardMixtureEulerSolver(
    model=wrapped_probability_denoiser,
    path=path,
    vocabulary_size=vocab_size,
    source_distribution_p=torch.full((vocab_size,), 1.0 / vocab_size, device=device),
)


def run_stages(token_paths, x_init, **kw):
    '''Chain token paths, e.g. the stages of sequential generation.'''
    x = x_init
    for p in token_paths:
        x = solver.sample(x_init=x, token_path=p, **kw)
    return x

## Sample from the trained model: unconditional (diagonal path)

In [ ]:
nfe = 64
step_size = 1 / nfe
n_samples = 200000

x_init = torch.randint(size=(n_samples, dim), high=vocab_size, device=device)
sol, snapshots = solver.sample(x_init=x_init, token_path=unconditional_path(dim), step_size=step_size, n_snapshots=8)

def heatmap(ax, x, title):
    x = x.cpu().numpy()
    H = np.histogram2d(x[:, 0], x[:, 1], bins=vocab_size, range=[[0, vocab_size], [0, vocab_size]])[0]
    cmax = max(np.quantile(H, 0.95), 1.0)
    ax.imshow(H.T, origin='lower', norm=cm.colors.Normalize(vmin=0.0, vmax=cmax))
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title)

fig, axs = plt.subplots(1, len(snapshots), figsize=(20, 3))
for idx, x_s in enumerate(snapshots):
    heatmap(axs[idx], x_s, f't = {idx / (len(snapshots) - 1):.2f}')
plt.tight_layout()
plt.show()
print(f"on-support fraction: {in_support(sol):.4f}   block TV: {block_tv(sol):.4f}")

## Every task from the same model

Two numbers per task:
* the fraction of samples on the checkerboard (target 1, uniform noise about 0.5);
* the total-variation distance of the 16 block masses from the truth (0 is perfect).

Unlike the continuous model, a discrete model can put exactly zero mass outside the support, so both can get close to their ideal values.

The **naive clamp** row uses the same model and the same clamped observed token, but feeds the diagonal time $(s,s)$, so the model is not told that the observed token is clean. That is how a scalar-time model has to inpaint. How much of the early mismatch the late steps fix depends on the model; train with `task_mix = (1.0, 0.0, 0.0)` to see it for your original diagonal-only model.

In [ ]:
torch.manual_seed(1)
n = 200000
z, x1 = torch.randint(size=(n, dim), high=vocab_size, device=device), inf_train_gen(vocab_size, n, device=device)
obsX = torch.tensor([1.0, 0.0], device=device)                                     # observe x, generate y
obsY = torch.tensor([0.0, 1.0], device=device)                                     # observe y, generate x
obsR = F.one_hot(torch.randint(2, (n,)), 2).float().to(device)                     # one random token per sample

def clamp(obs):
    return torch.where(obs.expand_as(x1).bool(), x1, z)

tasks = {
    "unconditional":         lambda **kw: solver.sample(x_init=z, token_path=unconditional_path(dim), **kw),
    "inpaint: x observed":   lambda **kw: solver.sample(x_init=clamp(obsX), token_path=inpainting_path(obsX), **kw),
    "inpaint: y observed":   lambda **kw: solver.sample(x_init=clamp(obsY), token_path=inpainting_path(obsY), **kw),
    "inpaint: random token": lambda **kw: solver.sample(x_init=clamp(obsR), token_path=inpainting_path(obsR), **kw),
    "sequential: x then y":  lambda **kw: run_stages(sequential_paths([0, 1], dim), z, **kw),
    "sequential: y then x":  lambda **kw: run_stages(sequential_paths([1, 0], dim), z, **kw),
}
corrector = 1.0
samples = {name: (fn(step_size=step_size), fn(step_size=step_size, corrector=corrector)) for name, fn in tasks.items()}
naive = solver.sample(x_init=clamp(obsX), token_path=inpainting_path(obsX), step_size=step_size,
                      time_fn=lambda s: s * torch.ones(dim, device=device))

print(f"{'task':24s} {'support':>8s} {'block TV':>9s} | {'+ corrector':>11s} {'block TV':>9s}")
for name, (a, b) in samples.items():
    print(f"{name:24s} {in_support(a):8.4f} {block_tv(a):9.4f} | {in_support(b):11.4f} {block_tv(b):9.4f}")
print(f"{'naive clamp: x observed':24s} {in_support(naive):8.4f} {block_tv(naive):9.4f}")
print(f"(target {in_support(x1):.4f}, uniform noise {in_support(z):.4f})")

for name, obs in [("inpaint: x observed", obsX), ("inpaint: y observed", obsY), ("inpaint: random token", obsR)]:
    a, b = samples[name]
    o = obs.expand_as(x1).bool()
    print(f"observed tokens changed, {name:22s}: {(a != x1)[o].sum().item()} (Euler), {(b != x1)[o].sum().item()} (+ corrector)")

In [ ]:
fig, axs = plt.subplots(2, len(samples) + 1, figsize=(4 * (len(samples) + 1), 8))
heatmap(axs[0, 0], x1, 'target')
heatmap(axs[1, 0], naive, f'naive clamp, x observed\non support {in_support(naive):.3f}')
for c, (name, (a, b)) in enumerate(samples.items(), start=1):
    heatmap(axs[0, c], a, f'{name}\non support {in_support(a):.3f}')
    heatmap(axs[1, c], b, f'{name} + corrector\non support {in_support(b):.3f}')
plt.tight_layout()
plt.show()

### Inpainting as conditional sampling: fix $x$, generate $y$

For a fixed observed $x$, the true $p(y\mid x)$ is uniform with mass $1/64$ on each of the 64 values in two of the four $y$-blocks.

In [ ]:
torch.manual_seed(2)
m = 50000
grid = np.arange(vocab_size)
fig, axs = plt.subplots(1, 4, figsize=(20, 3.5))
for ax, c in zip(axs, [16, 48, 80, 112]):
    init = torch.stack([torch.full((m,), c, device=device), torch.randint(0, vocab_size, (m,), device=device)], dim=1)
    y_mt = solver.sample(x_init=init, token_path=inpainting_path(obsX), step_size=step_size)[:, 1].cpu().numpy()
    y_nv = solver.sample(x_init=init, token_path=inpainting_path(obsX), step_size=step_size,
                         time_fn=lambda s: s * torch.ones(dim, device=device))[:, 1].cpu().numpy()
    true = ((((c // 32) + (grid // 32)) % 2) == 0) / 64
    ax.hist(y_mt, bins=vocab_size, range=(0, vocab_size), density=True, alpha=0.6, label='multitask (per-token time)')
    ax.hist(y_nv, bins=vocab_size, range=(0, vocab_size), density=True, alpha=0.4, label='naive clamp (diagonal time)')
    ax.step(grid, true, 'k', where='post', lw=1.2, label=r'true $p(y\mid x)$')
    ax.set_title(f"x observed = {c}")
    ax.legend(fontsize=7)
plt.show()

### The diagonal is your original model

On the diagonal $t^i=s$ this model is a scalar-time DFM model. Wrapped to take the library solver's `[batch]` time, it runs with the stock `MixtureDiscreteEulerSolver` unchanged, for unconditional generation only, since the stock solver has one time for all tokens.

In [ ]:
class DiagonalWrappedModel(ModelWrapper):
    def forward(self, x: torch.Tensor, t: torch.Tensor, **extras):
        return torch.softmax(self.model(x, t[:, None].expand_as(x), **extras), dim=-1)


stock_solver = MixtureDiscreteEulerSolver(model=DiagonalWrappedModel(probability_denoiser),
                                          path=MixtureDiscreteProbPath(scheduler=scheduler), vocabulary_size=vocab_size)
x_stock = stock_solver.sample(x_init=torch.randint(size=(n_samples, dim), high=vocab_size, device=device),
                              step_size=step_size, time_grid=torch.tensor([0.0, 1.0 - epsilon]))
print(f"stock solver, unconditional: on-support {in_support(x_stock):.4f}, block TV {block_tv(x_stock):.4f}")